In [ ]:
# SnowHealth Demand Forecasting - Infrastructure Setup
# This notebook verifies the database objects and explores the synthetic dataset.

import sys, os

# Find workspace source directory
work_dir = os.getcwd()
source_dir = os.path.join(work_dir, 'source')
if os.path.isdir(source_dir):
    sys.path.insert(0, source_dir)
    print(f"Source dir found: {source_dir}")
else:
    # Try alternative paths
    for candidate in ['/workspace/source', os.path.join(os.path.dirname(work_dir), 'source')]:
        if os.path.isdir(candidate):
            sys.path.insert(0, candidate)
            source_dir = candidate
            print(f"Source dir found: {candidate}")
            break
    else:
        print(f"CWD: {work_dir}")
        print(f"Dir contents: {os.listdir(work_dir)}")
        # List parent too
        print(f"Parent contents: {os.listdir(os.path.dirname(work_dir))}")
        raise FileNotFoundError("Cannot find source/ directory")

from snowflake.snowpark.context import get_active_session
import config as cfg

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_RAW_SCHEMA}").collect()
print(f"Connected to {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_RAW_SCHEMA}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Role: {session.get_current_role()}")

## Verify Database Objects
Confirm that the SNOWHEALTH_ML_DB database, schemas, and transaction table are created.

In [ ]:
%%sql -r schemas_result
SHOW SCHEMAS IN DATABASE SNOWHEALTH_ML_DB;

In [ ]:
%%sql -r data_summary
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(DISTINCT STORE_ID) AS NUM_STORES,
    COUNT(DISTINCT SKU_ID) AS NUM_SKUS,
    COUNT(DISTINCT REGION) AS NUM_REGIONS,
    COUNT(DISTINCT FULFILLMENT_CHANNEL) AS NUM_CHANNELS,
    COUNT(DISTINCT PRODUCT_CATEGORY) AS NUM_CATEGORIES,
    MIN(TRANSACTION_DATE) AS DATE_FROM,
    MAX(TRANSACTION_DATE) AS DATE_TO
FROM SNOWHEALTH_ML_DB.SNOWHEALTH_RAW.SNOWHEALTH_DEMAND_TRANSACTIONS;

In [ ]:
# Explore the raw data
import pandas as pd

df_raw = session.table(cfg.SNOWHEALTH_TRANSACTIONS_TABLE).to_pandas()
print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn types:")
print(df_raw.dtypes)
print(f"\nFirst 5 rows:")
df_raw.head()

In [ ]:
# Distribution of key dimensions
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_raw['REGION'].value_counts().plot(kind='bar', ax=axes[0,0], color='#cc0000')
axes[0,0].set_title('Transactions by Region', fontsize=12)
axes[0,0].tick_params(axis='x', rotation=0)

df_raw['PRODUCT_CATEGORY'].value_counts().plot(kind='barh', ax=axes[0,1], color='#cc0000')
axes[0,1].set_title('Transactions by Product Category', fontsize=12)

df_raw['FULFILLMENT_CHANNEL'].value_counts().plot(kind='bar', ax=axes[1,0], color='#cc0000')
axes[1,0].set_title('Transactions by Fulfillment Channel', fontsize=12)
axes[1,0].tick_params(axis='x', rotation=15)

axes[1,1].hist(df_raw['UNITS_SOLD'], bins=40, color='#cc0000', edgecolor='white')
axes[1,1].set_title('Distribution of Units Sold', fontsize=12)
axes[1,1].set_xlabel('Units Sold')

plt.suptitle('SnowHealth Demand Data - Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Time series demand trend
fig, ax = plt.subplots(figsize=(14, 5))

daily_demand = df_raw.groupby('TRANSACTION_DATE')['UNITS_SOLD'].sum().reset_index()
daily_demand['TRANSACTION_DATE'] = pd.to_datetime(daily_demand['TRANSACTION_DATE'])
daily_demand = daily_demand.sort_values('TRANSACTION_DATE')

ax.plot(daily_demand['TRANSACTION_DATE'], daily_demand['UNITS_SOLD'], color='#cc0000', alpha=0.5, linewidth=0.8)
rolling = daily_demand['UNITS_SOLD'].rolling(window=14).mean()
ax.plot(daily_demand['TRANSACTION_DATE'], rolling, color='#333333', linewidth=2, label='14-day Rolling Avg')
ax.set_title('Daily Demand Trend (All Products)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Units Sold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for numeric columns
print("=" * 60)
print("SNOWHEALTH DEMAND FORECASTING - SETUP COMPLETE")
print("=" * 60)
print(f"Database: {cfg.SNOWHEALTH_DATABASE}")
print(f"Schemas: {cfg.SNOWHEALTH_RAW_SCHEMA}, {cfg.SNOWHEALTH_FEATURES_SCHEMA}, {cfg.SNOWHEALTH_MODELS_SCHEMA}")
print(f"Rows: {len(df_raw):,}")
print(f"Date Range: {df_raw['TRANSACTION_DATE'].min()} to {df_raw['TRANSACTION_DATE'].max()}")
print(f"Stores: {df_raw['STORE_ID'].nunique()} | SKUs: {df_raw['SKU_ID'].nunique()}")
print(f"Regions: {df_raw['REGION'].nunique()} | Channels: {df_raw['FULFILLMENT_CHANNEL'].nunique()}")
print(f"Target (UNITS_SOLD): mean={df_raw['UNITS_SOLD'].mean():.1f}, std={df_raw['UNITS_SOLD'].std():.1f}")
print("=" * 60)
print("Proceed to 02_feature_engineering.ipynb")